# 实验 01：CIFAR-10 与完整分类流程
用一个较小的 CNN 理解完整流程，然后在后续实验替换网络结构。

阅读：[对应教材章节](https://zh.d2l.ai/chapter_computer-vision/image-augmentation.html)。图像数据统一为 CIFAR-10。
本实验默认是 32×32 小图教学适配版。原生 torchvision 版本的切换方法见后文。


In [ ]:
from pathlib import Path
import os, sys

# 从项目根目录、notebooks/ 或它们的子目录启动时都能找到 labkit。
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "labkit").is_dir()), None)
if ROOT is None:
    raise RuntimeError("请将整个课程目录解压后，从项目根目录启动 JupyterLab")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("项目根目录:", ROOT)


## 1. 实验配置
先用 `SMOKE=True` 跑通，再设为 False 正式训练。小规模检查不能代表模型最终性能。


In [ ]:
import inspect
from datetime import datetime
import torch
from labkit.common import choose_device, seed_everything
from labkit.data import load_image_data
from labkit.image_models import build_image_model, parameter_count
from labkit.engine import run_epoch, fit, final_test
from labkit.plots import show_images, plot_history, plot_confusion, save_run_plots

MODEL = "smallcnn"
VARIANT = "cifar"
PRETRAINED = False
FREEZE_BACKBONE = False
SMOKE = os.environ.get("LAB_SMOKE", "0") == "1"
SEED = 42
BATCH_SIZE = 64
EPOCHS = 1 if SMOKE else 20
LR = 0.01
USE_AMP = False  # 可在完成基础实验后改为 True，比较显存与速度。
DATA_DIR = ROOT / "data"
RUN_DIR = ROOT / "runs" / (MODEL + "_" + VARIANT + datetime.now().strftime("_%Y%m%d_%H%M%S_%f"))
seed_everything(SEED)
device = choose_device("auto")
print("device:", device, "SMOKE:", SMOKE, "结果目录:", RUN_DIR)


## 2. 数据加载、划分与预处理
官方训练集按类别分成 45,000 训练 / 5,000 验证，官方测试集保持独立。
训练图像使用随机裁剪与翻转，验证/测试只做固定预处理。
数据加载代码在 `labkit/data.py`，运行下面一格查看实际规模与张量形状。


In [ ]:
limits = [128, 64, 64] if SMOKE else [0, 0, 0]
bundle = load_image_data(DATA_DIR, batch_size=BATCH_SIZE, seed=SEED,
                         workers=0, variant=VARIANT, augment=True,
                         max_train=limits[0], max_val=limits[1], max_test=limits[2])
print(bundle.meta["sizes"])
images, labels = next(iter(bundle.val))
print("images:", images.shape, "labels:", labels.shape)
show_images(images, labels, bundle.meta["classes"], bundle.meta["mean"], bundle.meta["std"])


## 3. 创建模型
输出是 `[batch, 10]` 的 logits。`CrossEntropyLoss` 内部处理 log-softmax，训练前不手动 softmax。


In [ ]:
model = build_image_model(MODEL, VARIANT, PRETRAINED, FREEZE_BACKBONE).to(device)
print(model)
print("参数量:", parameter_count(model))
model.eval()
with torch.no_grad():
    print("输出形状:", model(images[:2].to(device)).shape)


## 4. 阅读完整训练循环
`run_epoch` 包含训练/评估模式切换、前向传播、损失、梯度、裁剪、参数更新与按样本统计指标。
`fit` 每个 epoch 只在验证集上选模型，最后恢复最佳权重。以下代码直接显示当前实现，可去源文件修改。


In [ ]:
print(inspect.getsource(run_epoch))
print(inspect.getsource(fit))


## 5. 训练与验证
先把模型移动到 GPU，再创建优化器。第一次正式实验建议保留相同划分和超参数。


In [ ]:
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
config = {"task": "image", "model": MODEL, "variant": VARIANT,
          "pretrained": PRETRAINED, "freeze_backbone": FREEZE_BACKBONE,
          "seed": SEED, "epochs": EPOCHS, "batch_size": BATCH_SIZE,
          "lr": LR, "amp": USE_AMP, "smoke": SMOKE,
          "parameters": parameter_count(model), "torch_version": str(torch.__version__)}
history = fit(model, bundle, optimizer, device, RUN_DIR, config,
               epochs=EPOCHS, amp=USE_AMP, grad_clip=5.0,
               freeze_backbone=FREEZE_BACKBONE, scheduler=scheduler)
plot_history(history)


## 6. 最终测试
此时模型已经恢复到最佳验证 epoch。测试结果用于最终报告，不能据此反复调参。混淆矩阵的行是真实类别，列是预测类别。


In [ ]:
metrics = final_test(model, bundle, device, RUN_DIR)
save_run_plots(history, metrics, RUN_DIR)
plot_confusion(metrics["confusion_matrix"], bundle.meta["classes"])
print("accuracy:", metrics["accuracy"], "macro F1:", metrics["macro_f1"])


## 7. 可视化预测
标题格式为“真实类别 / 预测类别”。对于自选图片，可使用项目中的 `predict.py`。


In [ ]:
images, labels = next(iter(bundle.test))
model.eval()
with torch.no_grad():
    predictions = model(images.to(device)).argmax(1).cpu()
show_images(images, labels, bundle.meta["classes"], bundle.meta["mean"],
             bundle.meta["std"], predictions=predictions)


## 8. 扩展：原生 torchvision 模型
重新运行实验前将 `VARIANT="torchvision"`、`PRETRAINED=True`，输入自动变为 224×224。
`FREEZE_BACKBONE=True` 只训练新分类层。AlexNet / ResNet18 建议 batch size 16，VGG11 建议先设为 8。
SmallCNN 不提供此模式。迁移学习实验单独比较，并报告额外使用了 ImageNet 预训练权重。

命令行入口示例：
```bash
python train_image.py --model resnet18 --variant torchvision --pretrained --freeze-backbone --batch-size 16 --epochs 5 --run-dir runs/resnet_transfer
```


## 9. 实操练习
1. 设为不做增广，与原配置对比验证曲线。
2. 把学习率改为 0.001 和 0.1，解释 loss 的变化。
3. 只观察训练准确率会遗漏什么问题？

实验报告按 `docs/REPORT_TEMPLATE.md` 填写。所有数值来自自己的实际运行。
